# LSTM text generation
We will be using a Sequence-to-Sequence (Seq2Seq) LSTM The input and output both are sequence not a single number unlike the finance example. The hidden state of all cells are considered in the loss calculation as intermediate outputs or hidden states are prediction of next word which comes after the input of the that partcular cell's input.

Link: https://www.kaggle.com/code/aritrase/text-generation-using-pytorchlstm

Before we start we will see two concepts.

## Gradient clipping

The recurrent nature of LSTM makes the gradient multiplicative in nature. This can lead to exploding or vanishing gradients. Gradient clipping is a technique to prevent exploding gradients such as `NAN` or `inf`. It is done by clipping the gradients to a threshold value.

## Word embeddings
NNs are mathematical functions, and they only understand number. So, if we are dealing with words or text data, we need to find a way to convert the text data into numbers, which is called word embedding. One way to do this is to use one-hot encoding. There are other fancy ways to do this in transformers such as word2vec, GloVe etc., but we will stick to one-hot encoding for now.
### One hot encoding


Say we have 5 unique words in our vocabulary. Each word can represented as a vector.
```
1 0 0 0 0 0 - I
0 1 0 0 0 0 - am
0 0 1 0 0 0 - going
0 0 0 1 0 0 - to
0 0 0 0 1 0 - office
```

This words well we have small vocabulary, but if we have a large vocabulary, then the one-hot encoding will be very sparse (a lot of zeros). One hot encoding has two main problems:

* consumes a lot of memory to store the words
* it does not capture the relationship between the words. Each word is orthogonal to each other for no reason.

### Word embeddings
Here we try to preserve the relationship between the words. Each attribute of a word is treated as a vector space. Words are represented as points (vectors) in this high-dimensional space.

Here are a number of sentences:

* The physicist runs to the store.
* The physicist majored in physics.
* The mathematician and physicist both like coffee.

This is how the word embedding should look like after the training.

$
q_{\text{mathematician}} = 
\begin{bmatrix}
\text{can run: } 2.3 \\ 
\text{likes coffee: } 9.4 \\ 
\text{majored in Physics: } -5.5 \\ 
\vdots
\end{bmatrix},
\quad
q_{\text{physicist}} = 
\begin{bmatrix}
\text{can run: } 2.5 \\ 
\text{likes coffee: } 9.1 \\ 
\text{majored in Physics: } 6.4 \\ 
\vdots
\end{bmatrix}
$

Here both the physicist and mathematicians "can run", so the scores are very close 2.3 and 2.5. Both like coffee, so the scores are very close 9.4 and 9.5. The physicist majored in physics, so the score is 6.4, but the mathematician didn't so his score is -5.5. This is how the word embeddings are trained.

### Cosine similarity
The similarity measure starts by calculating the dot product of the two vectors.

$\text{Similarity}(\text{physicist}, \text{mathematician}) = q_{\text{physicist}} \cdot q_{\text{mathematician}}$

To make the similarity independent of the magnitude (length) of the vectors, the dot product is normalized by dividing it by the product of the magnitudes (or norms) of the two vectors:

$\text{Similarity}(\text{physicist}, \text{mathematician}) = 
\frac{q_{\text{physicist}} \cdot q_{\text{mathematician}}}{\|q_{\text{physicist}}\| \cdot \|q_{\text{mathematician}}\|} = \cos(\phi)$

$\text{where } \phi \text{ is the angle between the two vectors.}$

$\|q\| = \sqrt{\sum_{i=1}^{n} q_i^2}$

* If the vectors point in the same direction ( $\phi = 0^\circ$ ), the similarity is  1.
* If they are orthogonal ( $\phi = 90^\circ$ ), the similarity is  0 .
* If they point in opposite directions ( $\phi = 180^\circ$ ), the similarity is  -1 .

In [46]:
import torch
import torch.nn as nn
import os
import numpy as np
from torch.nn.utils import clip_grad_norm_
import torch.nn.functional as F

In [47]:
# Check that MPS is available on Apple M series chip
if not torch.backends.mps.is_available():
    if not torch.backends.mps.is_built():
        print("MPS not available because the current PyTorch install was not "
              "built with MPS enabled.")
    else:
        print("MPS not available because the current MacOS version is not 12.3+ "
              "and/or you do not have an MPS-enabled device on this machine.")

else:
    device = torch.device("mps")
    print("MPS is available on this machine.")

MPS is available on this machine.


### Sentence indexing and embedding creation
This is how we created embeddings manually. Although, in PyTorch we use a class to automatically create embeddings.

In [48]:
word_to_idx = {"I": 0, "am": 1, "going": 2, "to": 3, "office": 4}
embeddings = nn.Embedding(len(word_to_idx), len(word_to_idx)-1).to(device) # 5 words, 4 dimensions or embedding size
embeddings.weight # entire embedding matrix initialised with random values

Parameter containing:
tensor([[ 0.4926,  0.3890, -1.3013,  0.4565],
        [ 0.2881, -0.9478, -1.0578,  0.2430],
        [-0.4723, -0.8501,  1.1287,  1.7975],
        [ 2.6841,  0.9581,  0.0057, -1.1785],
        [-0.1718, -0.0071,  2.1987, -1.0892]], device='mps:0',
       requires_grad=True)

### Word embedding
Word -> Index -> Embedding

In [49]:
word_index = torch.tensor([word_to_idx["going"]]).to(device)
print(word_index) # index of "going" is 2.
# Extract the embedding of the word "going"
going = embeddings(word_index)
print(going)

tensor([2], device='mps:0')
tensor([[-0.4723, -0.8501,  1.1287,  1.7975]], device='mps:0',
       grad_fn=<EmbeddingBackward0>)


### Dictionary class
The dictionary class is used to automate the process of creating a mapping between the words and the indices. It also has a method to convert the words to indices and vice versa. It generally follows the following structure:

Word -> Index -> Embedding

* `word2idx` - dictionary to convert words to indices
* `idx2word` - dictionary to convert indices to back to words

Dictionary class does below mentioned tasks:
* It checks if the passed word is already present in the dictionary or not.
* If it's a new word the class adds the word to the dictionary , assigns an index to the word.

In [50]:
class Dictionary(object):
    def __init__(self):
        self.word2idx = {}
        self.idx2word = {}
        self.idx = 0

    def add_word(self, word):
        if word not in self.word2idx:
            self.word2idx[word] = self.idx
            self.idx2word[self.idx] = word
            self.idx += 1
            
    def __len__(self):
        return len(self.word2idx)

### Corpus Class:
Corpus class with the help of Dictionary class doing the below operations -

* Creates an object of the Dictionary class which is to be used in future operations.
* In the get_data method below operations are going on -
    * Opens the file in read mode. This file will contain gramatically correction sentences for trainign and testing.
    * Reads the file line by line, splits each of the words in lines.
    * Adds a end of sentence tokoen at the end of each line. This helps in ending the sentence during generation/ inference.
    * Maintains a variable tokens to keep track of the total number of words.
    * Adds the words in the dictionary.
    * Once all the words are added in the dictonary creates a long tensor named 'ids'
    * In the 'ids' all the index from the dictionary is stored using word2idx.
    * Makes sure that all batches are of same size

In [51]:
class Corpus(object):
    
    def __init__(self):
        self.dictionary = Dictionary()

    def get_data(self, path, batch_size=20):
        # Open the file and read the words
        with open(path, 'r') as f:
            tokens = 0
            for line in f:
                words = line.split() + ['<eos>'] # <eos> is end of sentence token
                tokens += len(words)
                for word in words: 
                    self.dictionary.add_word(word)  # add the word to the dictionary
        # Create a 1-D tensor which contains index of all the words in the file with the help of word2idx
        ids = torch.LongTensor(tokens)
        token = 0
        with open(path, 'r') as f:
            for line in f:
                words = line.split() + ['<eos>']
                for word in words:
                    ids[token] = self.dictionary.word2idx[word] # get the index of the word
                    token += 1
        # no of required batches            
        num_batches = ids.shape[0] // batch_size     
        #Remove the remainder from the last batch , so that always batch size is constant
        ids = ids[:num_batches*batch_size]
        # return (batch_size,num_batches)
        ids = ids.view(batch_size, -1)
        return ids

### Setting the parameter values

In [52]:

embed_size = 128    # Embedding layer size , input to the LSTM
hidden_size = 1024  # Hidden size of LSTM units h_t and c_t
num_layers = 1      # no LSTMs stacked
num_epochs = 10     # total no of epochs
batch_size = 20     # batch size
seq_length = 100    # sequence length or number of LSTM cells
learning_rate = 0.002 # learning rate

In [53]:
# Load the dataset
corpus = Corpus() # helper objects
ids = corpus.get_data('text_data.txt', batch_size)

In [54]:
# ids tensors contain all the index of each words
print("[batch_size, num_batches]: ",ids.shape) # the size represents (batch_size, num_batches)

# What is the vocabulary size ?
vocab_size = len(corpus.dictionary)
print("Vocablury size: ", vocab_size)

[batch_size, num_batches]:  torch.Size([20, 46479])
Vocablury size:  10000


In [55]:
num_batches = ids.shape[1] // seq_length
print(num_batches)

464


### LTSM class (no staking)

In [56]:
class LSTM(nn.Module):
    def __init__(self, vocab_size, embed_size, hidden_size, num_layers):
        super(LSTM, self).__init__()
        self.embed = nn.Embedding(vocab_size, embed_size) # maps words to feature vectors
        # Embed_size is the input to the LSTM and output is hidden_size
        self.lstm = nn.LSTM(embed_size, hidden_size, num_layers, batch_first=True) # LSTM layer
        self.linear = nn.Linear(hidden_size, vocab_size) # final dense layers to convert high-dimensional features to vocab_size, so we can predict the next word
    
    def forward(self, x, h):
        # perform word embedding
        x = self.embed(x)

        out, (h, c) = self.lstm(x, h) # (input , hidden state)

        # Reshape the output to (batch_size*sequence_length, hidden_size)
        out = out.reshape(out.size(0)*out.size(1), out.size(2))

        # decode hidden states of all time steps
        out = self.linear(out)
        return out, (h, c)

In [57]:
model = LSTM(vocab_size, embed_size, hidden_size, num_layers).to(device) # compare this with finance.ipynb to see differences
model

LSTM(
  (embed): Embedding(10000, 128)
  (lstm): LSTM(128, 1024, batch_first=True)
  (linear): Linear(in_features=1024, out_features=10000, bias=True)
)

In [58]:
# Cross-entropy compares the predicted probability distribution with the true target (the correct word).
criterion = nn.CrossEntropyLoss() # loss function
optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)

### Training

In [59]:
# to Detach the Hidden and Cell states from previous history
def detach(states):
    return [state.detach() for state in states]

In [60]:
for epoch in range (num_epochs):
    # initial hidden (h_t) and cell states (c_t)
    states = (torch.zeros(num_layers, batch_size, hidden_size).to(device),
              torch.zeros(num_layers, batch_size, hidden_size).to(device)) # goes in model.forward
    
    for i in range(0, ids.size(1) - seq_length, seq_length):
        # move with seq length from the the starting index and move till - (ids.size(1) - seq_length)
        # Get mini-batch inputs and targets
        # Here the input is ith word and output is (i+1)th word.
        # Since LSTM has inputs sequence the input will be from i to i+seq_length and output will be from i+1 to i+1+seq_length
        # For example:
        # Input: I am going to office
        # Output: am going to office <eos>

        inputs = ids[:, i:i+seq_length].to(device)
        targets = ids[:, (i+1):(i+1)+seq_length].to(device)
        

        states = detach(states)

        outputs, states = model(inputs, states)
        loss = criterion(outputs, targets.reshape(-1))

        outputs,states = model(inputs, states)
        loss = criterion(outputs, targets.reshape(-1))

        model.zero_grad()
        loss.backward()
         
        #The gradients are clipped in the range [-clip_value, clip_value]. This is to prevent the exploding gradient problem
        clip_grad_norm_(model.parameters(), 0.5)
        optimizer.step()
              
        step = (i+1) // seq_length
        if step % 100 == 0:
            print ('Epoch [{}/{}], Loss: {:.4f}'.format(epoch+1, num_epochs, loss.item()))


Epoch [1/10], Loss: 9.2095
Epoch [1/10], Loss: 5.8996
Epoch [1/10], Loss: 5.4343
Epoch [1/10], Loss: 5.3104
Epoch [1/10], Loss: 5.2331
Epoch [2/10], Loss: 5.4092
Epoch [2/10], Loss: 4.9063
Epoch [2/10], Loss: 4.4761
Epoch [2/10], Loss: 4.6224
Epoch [2/10], Loss: 4.4665
Epoch [3/10], Loss: 4.7139
Epoch [3/10], Loss: 4.3046
Epoch [3/10], Loss: 3.9162
Epoch [3/10], Loss: 4.1023
Epoch [3/10], Loss: 3.8677
Epoch [4/10], Loss: 4.0774
Epoch [4/10], Loss: 3.8280
Epoch [4/10], Loss: 3.5047
Epoch [4/10], Loss: 3.6535
Epoch [4/10], Loss: 3.3813
Epoch [5/10], Loss: 3.5651
Epoch [5/10], Loss: 3.4521
Epoch [5/10], Loss: 3.1813
Epoch [5/10], Loss: 3.2574
Epoch [5/10], Loss: 3.0016
Epoch [6/10], Loss: 3.1511
Epoch [6/10], Loss: 3.1675
Epoch [6/10], Loss: 2.8279
Epoch [6/10], Loss: 2.9487
Epoch [6/10], Loss: 2.6942
Epoch [7/10], Loss: 2.7763
Epoch [7/10], Loss: 2.8365
Epoch [7/10], Loss: 2.5980
Epoch [7/10], Loss: 2.6856
Epoch [7/10], Loss: 2.4416
Epoch [8/10], Loss: 2.5754
Epoch [8/10], Loss: 2.6240
E

### Generate new Text using the training model

In [63]:
# Test the model
# We will use the a random word from the dictionary as seed and generate next 500 words.
with torch.no_grad():
    with open('results.txt', 'w') as f:
        #intial hidden ane cell states
        state = (torch.zeros(num_layers, 1, hidden_size).to(device),
                 torch.zeros(num_layers, 1, hidden_size).to(device))
        
        # Pickup a random word from the dictionary
        # Select one word id randomly and convert it to shape (1,1)
        input = torch.randint(0,vocab_size, (1,)).long().unsqueeze(1).to(device)
                            # (min , max , shape) , convert to long tensor and make it a shape of 1,1 
        # reusing the output and state as input 500 times to generate 500 words.
        for i in range(500):
            output, _ = model(input, state)

            
            # Sample a word id from the exponential of the output 
            prob = output.exp()
            # pickup one word based on the probability
            word_id = torch.multinomial(prob, num_samples=1).item()
            #print(word_id)

            
            # Replace the input with sampled word id for the next time step
            input.fill_(word_id)

            
            word = corpus.dictionary.idx2word[word_id]
            # Write the results to file
            word = '\n' if word == '<eos>' else word + ' '
            f.write(word)

            
            if (i+1) % 100 == 0:
                print('Sampled [{}/{}] words and save to {}'.format(i+1, 500, 'results.txt'))

Sampled [100/500] words and save to results.txt
Sampled [200/500] words and save to results.txt
Sampled [300/500] words and save to results.txt
Sampled [400/500] words and save to results.txt
Sampled [500/500] words and save to results.txt


In [65]:
!cat results.txt

carolina upheld refuse machines dedicated described a government entities cuba are being robbed <unk> secret in the use statute columbus flow eagerness not yet modernization disproportionate aid banco exterior reduction out of the emergency crews allied-signal dreams preparation booklets spurred by holding company acknowledges that middle marketing already cut off book value lufthansa officers insist maidenform conference will serve earlier requests from eroding huge hidden spending restraint northern california plant genetic engineering company leases say each limits consent solicitation complained bitterly billionaire tourism believed that leave alone profit projection visitors constitute N million square ruling determine whether freight transport accused fees receives different letters horses noted that if a <unk> recorded monday prevented a lot fuel N million dollars us$ N million dollars us$ N million square foot democrats known as a larger test scores imf text them therefore sayi